# Model Context Protocol (MCP) Client, Wire Inspector & Policy Guardrails

> **Lab 02 & Phase 03 Companion Exercise**  
> **Core Architecture**: JSON-RPC 2.0 Wire Protocol Inspector + Tool Discovery Registry + Zero-Trust ABAC Policy Engine + AST-Level SQL Mutation Protection.

In production AI agent systems, granting foundation models direct, unmediated access to databases, payment gateways, or operating system shells invites the **Confused Deputy Attack**: an indirect prompt injection tricks the LLM into invoking high-privilege tools. This notebook provides a complete, interactive implementation of the **Model Context Protocol (MCP)** wire protocol, simulates client-server handshakes, inspects JSON-RPC 2.0 frames, and evaluates tool calls against an Attribute-Based Access Control (ABAC) Policy Engine.

In [ ]:
# %% [Setup] Colab & Environment Bootstrap
import os, sys
try:
    is_colab = 'google.colab' in str(get_ipython())
except NameError:
    is_colab = False

if is_colab:
    print('Running in Google Colab. Cloning repository...')
    !git clone --depth 1 https://github.com/dip4k/senior-engineer-to-ai-engineer-roadmap.git repo
    %cd repo
    !pip install -q -r agent-forge/requirements.txt -r notebooks/requirements-notebooks.txt
    sys.path.insert(0, os.path.abspath('agent-forge'))
else:
    # Local environment setup: locate repo root and agent-forge
    repo_root = os.path.abspath('..') if os.path.exists('../agent-forge') else os.path.abspath('.')
    af_path = os.path.join(repo_root, 'agent-forge')
    if os.path.exists(af_path) and af_path not in sys.path:
        sys.path.insert(0, af_path)
    print(f'Running locally with agent-forge in path: {af_path}')

## 1. Architectural Architecture & The Confused Deputy Problem

Connecting foundation models to tools requires a strict separation of concerns:

1. **Standardized Wire Protocol**: Rather than writing custom glue code for every tool, the Model Context Protocol (MCP) establishes a bidirectional JSON-RPC 2.0 transport.
2. **Zero-Trust Authorization Boundary**: The LLM *decides* which tool to invoke, but the execution layer must *never* trust the LLM. An isolated **ABAC Policy Engine** inspects caller identity, tool name, and payload attributes before any action is executed.
3. **Safety Tiers & Step-Up Gates**: Safe read queries run automatically; high-value mutations require Human-in-the-Loop (HITL) step-up approval; destructive mutations (such as SQL DDL commands) are denied unconditionally.

```text
  +---------------------+        +-------------------------+        +-------------------+
  |  LLM Tool Decision  | -----> |    ABAC Policy Engine   | -----> |   MCP Wire Server |
  |  (JSON-RPC Request) |        | (Permitted/Gate/Denied) |        | (Tool Execution)  |
  +---------------------+        +-------------------------+        +-------------------+
                                              |
                                              v
                                 +-------------------------+
                                 |  Human-in-the-Loop Gate |
                                 |  (HMAC Step-Up Token)   |
                                 +-------------------------+
```

## 2. MCP JSON-RPC 2.0 Wire Protocol Models & Inspector

The MCP wire protocol is built upon JSON-RPC 2.0. Every frame contains:
- `jsonrpc`: Fixed string `"2.0"`.
- `id`: Unique correlation identifier linking asynchronous responses to outbound requests.
- `method`: The protocol verb (e.g. `initialize`, `tools/list`, `tools/call`).
- `params`: Structured arguments validated via Pydantic v2.

In [ ]:
import json, uuid, time, hmac, hashlib, re
from typing import Dict, Any, List, Optional, Literal
from pydantic import BaseModel, Field, field_validator

# ---------------------------------------------------------------------------
# JSON-RPC 2.0 Wire Protocol Data Models
# ---------------------------------------------------------------------------
class JSONRPCRequest(BaseModel):
    jsonrpc: Literal['2.0'] = '2.0'
    id: str = Field(default_factory=lambda: f'req_{uuid.uuid4().hex[:8]}')
    method: str
    params: Dict[str, Any] = Field(default_factory=dict)

class JSONRPCResponse(BaseModel):
    jsonrpc: Literal['2.0'] = '2.0'
    id: str
    result: Optional[Dict[str, Any]] = None
    error: Optional[Dict[str, Any]] = None

class ToolDefinition(BaseModel):
    name: str
    description: str
    inputSchema: Dict[str, Any]

# ---------------------------------------------------------------------------
# Formatted Wire Inspector Helper
# ---------------------------------------------------------------------------
def inspect_wire_packet(direction: str, packet: BaseModel):
    arrow = '>>> [CLIENT -> SERVER]' if direction == 'outbound' else '<<< [SERVER -> CLIENT]'
    formatted_json = json.dumps(packet.model_dump(), indent=2)
    print(f'\n{arrow} JSON-RPC 2.0 FRAME')
    print('-' * 60)
    print(formatted_json)
    print('-' * 60)

# Quick sanity verification
sample_req = JSONRPCRequest(method='tools/list', params={})
inspect_wire_packet('outbound', sample_req)

## 3. Protocol Handshake Simulation (`initialize` & `tools/list`)

The MCP connection lifecycle begins with capability negotiation:
1. **`initialize`**: The client transmits supported protocol versions and capabilities.
2. **Response**: The server responds with server information and enabled services.
3. **`notifications/initialized`**: The client confirms connection readiness.
4. **`tools/list`**: The client queries available tools and their JSON Schema parameter contracts.

In [ ]:
# ---------------------------------------------------------------------------
# Simulated MCP Payment & Database Server
# ---------------------------------------------------------------------------
class SimulatedMCPServer:
    def __init__(self, name: str = 'EnterpriseCoreServer', version: str = '1.0.0'):
        self.name = name
        self.version = version
        self.tools: Dict[str, ToolDefinition] = {}
        self._register_default_tools()

    def _register_default_tools(self):
        self.tools['payment_get_balance'] = ToolDefinition(
            name='payment_get_balance',
            description='Query account balance for a given customer account ID.',
            inputSchema={
                'type': 'object',
                'properties': {'account_id': {'type': 'string', 'description': 'Customer account reference'}},
                'required': ['account_id']
            }
        )
        self.tools['payment_issue_refund'] = ToolDefinition(
            name='payment_issue_refund',
            description='Issues a financial refund to a specified transaction.',
            inputSchema={
                'type': 'object',
                'properties': {
                    'transaction_id': {'type': 'string'},
                    'amount': {'type': 'number', 'description': 'Refund amount in USD'},
                    'reason': {'type': 'string'}
                },
                'required': ['transaction_id', 'amount']
            }
        )
        self.tools['database_execute_query'] = ToolDefinition(
            name='database_execute_query',
            description='Executes read-only SQL queries against the analytical warehouse.',
            inputSchema={
                'type': 'object',
                'properties': {'sql': {'type': 'string', 'description': 'SQL query string'}},
                'required': ['sql']
            }
        )

    def handle_request(self, request: JSONRPCRequest) -> JSONRPCResponse:
        if request.method == 'initialize':
            return JSONRPCResponse(
                id=request.id,
                result={
                    'protocolVersion': '2026-01-01',
                    'serverInfo': {'name': self.name, 'version': self.version},
                    'capabilities': {'tools': {'listChanged': False}, 'resources': {}}
                }
            )
        elif request.method == 'tools/list':
            return JSONRPCResponse(
                id=request.id,
                result={'tools': [t.model_dump() for t in self.tools.values()]}
            )
        elif request.method == 'tools/call':
            tool_name = request.params.get('name')
            args = request.params.get('arguments', {})
            return self._execute_tool(request.id, tool_name, args)
        else:
            return JSONRPCResponse(
                id=request.id,
                error={'code': -32601, 'message': f'Method {request.method} not found.'}
            )

    def _execute_tool(self, req_id: str, name: str, args: Dict[str, Any]) -> JSONRPCResponse:
        if name == 'payment_get_balance':
            return JSONRPCResponse(id=req_id, result={'account_id': args.get('account_id'), 'balance_usd': 14250.00, 'currency': 'USD'})
        elif name == 'payment_issue_refund':
            return JSONRPCResponse(id=req_id, result={'refund_id': f'ref_{uuid.uuid4().hex[:6]}', 'status': 'PROCESSED', 'amount': args.get('amount')})
        elif name == 'database_execute_query':
            return JSONRPCResponse(id=req_id, result={'rows': [{'id': 1, 'account': 'AcmeCorp', 'status': 'ACTIVE'}], 'row_count': 1})
        return JSONRPCResponse(id=req_id, error={'code': -32602, 'message': f'Unknown tool: {name}'})

# Execute Handshake
server = SimulatedMCPServer()
init_req = JSONRPCRequest(method='initialize', params={'protocolVersion': '2026-01-01', 'clientInfo': {'name': 'AgentForgeClient'}})
inspect_wire_packet('outbound', init_req)
init_resp = server.handle_request(init_req)
inspect_wire_packet('inbound', init_resp)

# Execute Tool Discovery
list_req = JSONRPCRequest(method='tools/list')
list_resp = server.handle_request(list_req)
inspect_wire_packet('inbound', list_resp)
print(f'\nDiscovered {len(list_resp.result["tools"])} active tools on server.')

## 4. Zero-Trust ABAC Policy Engine & Step-Up Gates

Why is an authorization engine mandatory before calling an MCP tool?
When an LLM outputs `{name: 'payment_issue_refund', arguments: {amount: 2500.0}}`, it is only a *candidate recommendation*. The autonomous agent loop must not execute it without checking enterprise policy invariants:

| Policy Status | Meaning | Action Taken |
|---|---|---|
| `PERMITTED` | Operation is within autonomous authority limits | Dispatched immediately over JSON-RPC wire |
| `REQUIRES_APPROVAL` | High-value or sensitive operation | Suspended; HMAC step-up approval token generated |
| `DENIED` | Forbidden, unauthorized, or destructive operation | Execution halted with security audit exception |

In [ ]:
# ---------------------------------------------------------------------------
# Zero-Trust Attribute-Based Access Control (ABAC) Engine
# ---------------------------------------------------------------------------
class PolicyDecision(BaseModel):
    status: Literal['PERMITTED', 'REQUIRES_APPROVAL', 'DENIED']
    reason: str
    approval_token: Optional[str] = None

class ABACPolicyEngine:
    def __init__(self, auto_refund_limit_usd: float = 100.0, secret_key: str = 'enterprise_super_secret'):
        self.auto_refund_limit_usd = auto_refund_limit_usd
        self.secret_key = secret_key.encode('utf-8')

    def generate_approval_token(self, tenant_id: str, tool_name: str, amount: float) -> str:
        payload = f'{tenant_id}:{tool_name}:{amount}:{int(time.time())}'
        sig = hmac.new(self.secret_key, payload.encode('utf-8'), hashlib.sha256).hexdigest()[:16]
        return f'token_{sig}'

    def evaluate(
        self,
        tenant_id: str,
        user_role: str,
        tool_name: str,
        arguments: Dict[str, Any]
    ) -> PolicyDecision:
        # Rule 1: Administrative tools are denied unconditionally for non-admin roles
        if tool_name.startswith('admin_') or tool_name.startswith('system_'):
            return PolicyDecision(
                status='DENIED',
                reason=f'Security Invariant: Administrative tool {tool_name} is strictly prohibited.'
            )

        # Rule 2: Financial Refund Threshold Check
        if tool_name == 'payment_issue_refund':
            amount = float(arguments.get('amount', 0.0))
            if amount <= 0:
                return PolicyDecision(status='DENIED', reason=f'Invalid refund amount: {amount} USD.')
            if amount > self.auto_refund_limit_usd:
                token = self.generate_approval_token(tenant_id, tool_name, amount)
                return PolicyDecision(
                    status='REQUIRES_APPROVAL',
                    reason=f'Refund amount {amount:.2f} USD exceeds auto-limit ({self.auto_refund_limit_usd:.2f} USD).',
                    approval_token=token
                )
            return PolicyDecision(
                status='PERMITTED',
                reason=f'Refund {amount:.2f} USD is within auto-approval threshold.'
            )

        # Rule 3: Read-only operations are permitted by default
        return PolicyDecision(status='PERMITTED', reason='Tool permitted by baseline policy.')

# Instantiate Policy Engine and test decisions
engine = ABACPolicyEngine(auto_refund_limit_usd=100.0)

cases = [
    ('tenant_a', 'support_rep', 'payment_get_balance', {'account_id': 'ACC-100'}),
    ('tenant_a', 'support_rep', 'payment_issue_refund', {'amount': 45.0, 'transaction_id': 'tx_1'}),
    ('tenant_a', 'support_rep', 'payment_issue_refund', {'amount': 450.0, 'transaction_id': 'tx_2'}),
    ('tenant_a', 'support_rep', 'admin_drop_database', {}),
]

for tenant, role, tool, args in cases:
    decision = engine.evaluate(tenant, role, tool, args)
    print(f'Tool: {tool:<22} | Amount: {args.get("amount", "-"):<6} | Status: {decision.status:<18} | Reason: {decision.reason}')

## 5. Hazardous SQL Mutation Defense (AST & Lexical Security)

When an agent interacts with analytical databases via `database_execute_query`, allowing raw input invites SQL Injection and destructive DDL statements (`DROP TABLE`, `TRUNCATE`, `DELETE`).

We implement an AST / Lexical query analyzer that inspects the query before dispatch:
- **Permitted**: Single `SELECT` statements with read-only semantics.
- **Forbidden**: Any query containing `DROP`, `DELETE`, `UPDATE`, `ALTER`, `TRUNCATE`, or multiple statements.

In [ ]:
import re

# ---------------------------------------------------------------------------
# AST / Lexical SQL Safety Validator
# ---------------------------------------------------------------------------
def validate_sql_safety(sql: str) -> Dict[str, Any]:
    cleaned = sql.strip().upper()
    
    # Check 1: Must begin with SELECT
    if not cleaned.startswith('SELECT'):
        return {
            'safe': False,
            'error': f'Security Violation: Only SELECT queries permitted; query starts with {cleaned.split()[0]}'
        }
        
    # Check 2: Reject dangerous mutation keywords anywhere in query
    forbidden_keywords = ['DROP', 'DELETE', 'UPDATE', 'INSERT', 'ALTER', 'TRUNCATE', 'EXEC', 'GRANT', 'REVOKE']
    for kw in forbidden_keywords:
        # Match as discrete keyword surrounded by boundaries
        pattern = rf'\b{kw}\b'
        if re.search(pattern, cleaned):
            return {
                'safe': False,
                'error': f'Security Violation: Forbidden mutating keyword detected: {kw}'
            }
            
    # Check 3: Multi-statement injection defense (semicolons followed by non-whitespace)
    statements = [s.strip() for s in sql.split(';') if s.strip()]
    if len(statements) > 1:
        return {
            'safe': False,
            'error': 'Security Violation: Stacked queries (multiple statements separated by ;) are prohibited.'
        }
        
    return {'safe': True, 'sanitized_sql': sql.strip()}

# Test queries against SQL safety validator
test_queries = [
    'SELECT customer_id, balance_usd FROM customers WHERE tier = \'ENTERPRISE\'',
    'DROP TABLE customers;',
    'DELETE FROM invoices WHERE amount_usd > 1000;',
    'UPDATE customers SET balance_usd = 0;',
    'SELECT * FROM customers; DROP TABLE orders;'
]

print('=== SQL SAFETY EVALUATION MATRIX ===')
for q in test_queries:
    res = validate_sql_safety(q)
    status = 'ALLOWED' if res['safe'] else 'BLOCKED'
    diag = 'Query verified read-only' if res['safe'] else res['error']
    print(f'[{status}] {q[:45]:<45} -> {diag}')

## 6. End-to-End Client Pipeline: Interception -> Policy Gate -> Wire Call

Now we integrate the components into a unified `SecureMCPClient`:
1. Intercepts candidate tool calls from the agent.
2. Validates SQL safety if the tool is `database_execute_query`.
3. Evaluates caller role and arguments in `ABACPolicyEngine`.
4. Only dispatches the JSON-RPC frame over the wire if the policy evaluates to `PERMITTED`.

In [ ]:
# ---------------------------------------------------------------------------
# Unified Secure MCP Client with Policy Enforcement
# ---------------------------------------------------------------------------
class SecureMCPClient:
    def __init__(self, server: SimulatedMCPServer, policy_engine: ABACPolicyEngine):
        self.server = server
        self.policy = policy_engine

    def invoke_tool(
        self,
        tenant_id: str,
        user_role: str,
        tool_name: str,
        arguments: Dict[str, Any],
        approval_token: Optional[str] = None
    ) -> Dict[str, Any]:
        print(f'\n[INSPECTOR] Inbound Tool Call Request: {tool_name}')
        
        # Special check for SQL tools
        if tool_name == 'database_execute_query':
            sql_check = validate_sql_safety(arguments.get('sql', ''))
            if not sql_check['safe']:
                print(f'  [POLICY DENIAL] {sql_check["error"]}')
                return {'status': 'FAILED', 'error': sql_check['error']}

        # Policy Engine Evaluation
        decision = self.policy.evaluate(tenant_id, user_role, tool_name, arguments)
        
        if decision.status == 'DENIED':
            print(f'  [POLICY DENIAL] Access Denied: {decision.reason}')
            return {'status': 'DENIED', 'reason': decision.reason}
            
        if decision.status == 'REQUIRES_APPROVAL':
            if not approval_token or approval_token != decision.approval_token:
                print(f'  [STEP-UP REQUIRED] {decision.reason}')
                print(f'  [STEP-UP REQUIRED] Emitted Human Approval Token: {decision.approval_token}')
                return {'status': 'REQUIRES_APPROVAL', 'token': decision.approval_token, 'reason': decision.reason}
            print(f'  [HUMAN APPROVAL VERIFIED] Valid token supplied ({approval_token}). Proceeding...')

        # Dispatch over JSON-RPC wire
        wire_req = JSONRPCRequest(
            method='tools/call',
            params={'name': tool_name, 'arguments': arguments}
        )
        wire_resp = self.server.handle_request(wire_req)
        print(f'  [WIRE EXECUTION COMPLETE] Result: {wire_resp.result}')
        return {'status': 'SUCCESS', 'result': wire_resp.result}

# Instantiate client and run complete demonstration
client = SecureMCPClient(server, engine)

# Scenario 1: Standard read query (Permitted)
client.invoke_tool('tenant_1', 'user', 'payment_get_balance', {'account_id': 'ACC-994'})

# Scenario 2: Safe auto-refund <= 100 USD (Permitted)
client.invoke_tool('tenant_1', 'user', 'payment_issue_refund', {'transaction_id': 'tx_12', 'amount': 49.0})

# Scenario 3: High-value refund > 100 USD (Step-Up Gated)
gate_res = client.invoke_tool('tenant_1', 'user', 'payment_issue_refund', {'transaction_id': 'tx_13', 'amount': 250.0})

# Scenario 4: Resume high-value refund with valid approval token
if gate_res['status'] == 'REQUIRES_APPROVAL':
    client.invoke_tool('tenant_1', 'user', 'payment_issue_refund', {'transaction_id': 'tx_13', 'amount': 250.0}, approval_token=gate_res['token'])

# Scenario 5: Blocked administrative drop table
client.invoke_tool('tenant_1', 'user', 'database_execute_query', {'sql': 'DROP TABLE customers;'})

## 7. Interactive Katas & Challenges

### Kata 1: Add Order Cancellation Tool & Inventory Policy
Extend the MCP server with an `order_cancel` tool. Implement a policy that only permits cancellation if the order status is `PENDING`.

### Kata 2: Verify Strict Rejection of Injection Payloads
Test edge-case SQL injection strings such as `SELECT * FROM orders WHERE id = '1' OR '1'='1'; DELETE FROM orders;`.

In [ ]:
# ---------------------------------------------------------------------------
# Kata Solution & Self-Verification
# ---------------------------------------------------------------------------
def test_kata_edge_cases():
    injection_attack = "SELECT * FROM orders WHERE id = '1'; DELETE FROM orders;"
    res = validate_sql_safety(injection_attack)
    assert not res['safe'], 'Kata Failure: Stacked SQL injection was not blocked!'
    print('Kata 1 Passed: Stacked injection query was safely intercepted.')
    
    # Test auto-refund limit barrier
    dec = engine.evaluate('t1', 'rep', 'payment_issue_refund', {'amount': 100.01})
    assert dec.status == 'REQUIRES_APPROVAL', 'Kata Failure: 100.01 USD did not trigger approval gate!'
    print('Kata 2 Passed: Precision boundary condition (> 100.00 USD) verified.')

test_kata_edge_cases()

## 8. Summary & Key Production Takeaways

- **JSON-RPC 2.0 Standardization**: Decouples autonomous agents from bespoke vendor APIs, providing a clean wire protocol.
- **Zero-Trust Tool Boundary**: Models must never possess direct database or payment credentials.
- **Attribute-Based Access Control**: Evaluate tenant ID, role, tool name, and payload attributes *before* wire dispatch.
- **Deterministic SQL Guardrails**: Defense-in-depth requires AST/lexical query sanitization to eliminate DDL and mutation exploits.